### Bronze Layer

there are customer_zip_code_prefix values that are not present in the geolocation table

In [0]:
%sql
select 
    * 
from bronze.customers c
left join bronze.geolocation g
on c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
where g.geolocation_zip_code_prefix is null

the geolocation table feels useless as the the location data is all present in the customers and sellers table already

In [0]:
%sql
select *
from (
    select 
        *,
        row_number() over(partition by geolocation_zip_code_prefix order by geolocation_zip_code_prefix) rn
    from bronze.geolocation
)
where rn>1

all orders_ids have their respective customer_ids... No null values present

In [0]:
%sql
select * from bronze.customers c
right join bronze.orders o
on c.customer_id = o.customer_id
where c.customer_id is null

all payments have an order_id to get mapped to

In [0]:
%sql
select 
    * 
from bronze.orders o
right join bronze.order_payments p
on o.order_id = p.order_id
where o.order_id is null

every order_item is associated with an order_id

In [0]:
%sql
select 
    * 
from bronze.orders o
right join bronze.order_items i
on o.order_id = i.order_id
where o.order_id is null

seller_id is properly mapped

In [0]:
%sql
select * from bronze.order_items i
right join bronze.sellers s
on i.seller_id = s.seller_id
where i.seller_id is null

4,938 reviews have no order_id mapped to... reviews are given to missing orders?? <br>
<b>Action</b>: Remove them completely

In [0]:
%sql
select * from bronze.orders o
right join bronze.order_reviews r
on o.order_id = r.order_id
where o.order_id is null

all products have been mapped to their corresponding orders

In [0]:
%sql
select * from bronze.order_items i
right join bronze.products p
on i.product_id = p.product_id
where i.product_id is null

### Silver Layer

all reviews have a corresponding order present in the orders table

In [0]:
%sql
select order_id, r.* from silver.fact_orders o
right join silver.dim_reviews r
using (review_key)
where o.order_id is null

In [0]:
%sql
select review_key
from silver.dim_reviews
where review_answer_timestamp is null

In [0]:
%sql
select * from silver.dim_products
join bronze.order_items
using (product_id)
where product_height_cm is null

In [0]:
%sql
select distinct product_category_name from silver.dim_products

In [0]:
%sql
select count(*) from silver.dim_payments where payment_value is null